In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

class EngineeringDataPipeline:
    """
    A robust Python-based data analytics pipeline designed for evaluating 
    Solid Mechanics & Stress Analysis data (Topic: SOL-05: Factor of Safety Variance).
    """
    
    def __init__(self, file_path: str):
        """
        Initializes the pipeline, maps internal tracking attributes, 
        and auto-creates mandatory output directories matching GitHub standards.
        """
        self.file_path = file_path
        self.raw_data = None
        self.cleaned_data = None
        self.analysis_results = {}
        
        # Ensure mandatory folders exist per project submission guidelines
        os.makedirs("data", exist_ok=True)
        os.makedirs("outputs", exist_ok=True)

    def ingest_data(self) -> pd.DataFrame:
        """
        MODULE 1: DATA INGESTION
        Safely loads the engineering CSV dataset with robust try-except error handling.
        """
        print("[INFO] Starting Data Ingestion Module...")
        try:
            if not os.path.exists(self.file_path):
                raise FileNotFoundError(f"Target data file at '{self.file_path}' could not be located.")
            
            # Load dataset and immediately clone it to the raw data repository folder
            self.raw_data = pd.read_csv(self.file_path)
            self.raw_data.to_csv("data/dataset_original.csv", index=False)
            print(f"[SUCCESS] Ingested {len(self.raw_data)} rows from raw source.")
            return self.raw_data
            
        except FileNotFoundError as fnf_err:
            print(f"[ERROR] Data Ingestion terminated: {fnf_err}")
            raise
        except Exception as e:
            print(f"[ERROR] An unexpected error occurred during ingestion: {e}")
            raise

    def clean_data(self, target_material: str = None) -> pd.DataFrame:
        """
        MODULE 2: AUTOMATED DATA CLEANING & UNIQUE FILTERING
        Handles missing values, drops duplicates, corrects data types, 
        applies a unique filter slice, and computes the engineering target (FoS).
        """
        print("[INFO] Starting Data Cleaning & Unique Filtering Module...")
        if self.raw_data is None:
            raise ValueError("No data found in buffer. Please run ingest_data() first.")
            
        try:
            # 1. Purge duplicate rows
            df = self.raw_data.drop_duplicates()
            
            # 2. Automated handling of missing/null values via column-wise median imputation
            numerical_cols = df.select_dtypes(include=[np.number]).columns
            for col in numerical_cols:
                if df[col].isnull().any():
                    median_value = df[col].median()
                    df[col] = df[col].fillna(median_value)
            
            # 3. Explicit Data Type Correction to prevent computational mismatching
            critical_features = ['Yield_Strength', 'Applied_Stress']
            for col in critical_features:
                if col in df.columns:
                    df[col] = df[col].astype(float)
            
            # 4. Mandatory Unique Filter Logic (Satisfies the 'No Sharing' rule)
            # Isolates a single material slice so your dataset calculation matrix is uniquely yours
            if target_material and 'Material_Type' in df.columns:
                df = df[df['Material_Type'] == target_material].copy()
                print(f"[FILTER] Programmatic unique filter applied: Material_Type == '{target_material}'")
            elif 'Material_Type' in df.columns:
                unique_groups = df['Material_Type'].unique()
                if len(unique_groups) > 0:
                    df = df[df['Material_Type'] == unique_groups[0]].copy()
                    print(f"[FILTER] Fallback programmatic filter applied: Material_Type == '{unique_groups[0]}'")
            
            # 5. Core Mathematical Transformation: Calculate Factor of Safety (FoS)
            # FoS = Yield Strength / Applied Operational Stress
            if 'Yield_Strength' in df.columns and 'Applied_Stress' in df.columns:
                df['Factor_of_Safety'] = df['Yield_Strength'] / df['Applied_Stress']
            else:
                raise KeyError("Missing core stress-strain metrics ('Yield_Strength' or 'Applied_Stress') in data.")

            self.cleaned_data = df
            # Save the clean slice to the mandatory data folder
            self.cleaned_data.to_csv("data/dataset_cleaned.csv", index=False)
            print(f"[SUCCESS] Cleaning finished. Active evaluation slice contains {len(self.cleaned_data)} records.")
            return self.cleaned_data
            
        except KeyError as k_err:
            print(f"[ERROR] Data cleaning failed due to structural column mismatch: {k_err}")
            raise
        except Exception as e:
            print(f"[ERROR] Critical pipeline exception caught during data cleaning: {e}")
            raise

    def analyze_data(self) -> dict:
        """
        MODULE 3: ENGINEERING DATA ANALYTICS (Strict NumPy Integration)
        Computes analytical indicators, outlier spreads, and correlations via NumPy vectors.
        """
        print("[INFO] Starting Engineering Data Analytics Module...")
        if self.cleaned_data is None:
            raise ValueError("Cleaned data array unpopulated. Please run clean_data() first.")
            
        try:
            # Cast pandas series into native, memory-contiguous NumPy arrays
            fos_array = self.cleaned_data['Factor_of_Safety'].to_numpy()
            stress_array = self.cleaned_data['Applied_Stress'].to_numpy()
            
            # 1. Descriptive Statistics using explicit NumPy array methods
            self.analysis_results['mean_fos'] = float(np.mean(fos_array))
            self.analysis_results['median_fos'] = float(np.median(fos_array))
            self.analysis_results['std_dev_fos'] = float(np.std(fos_array, ddof=1)) # Sample Standard Deviation
            self.analysis_results['variance_fos'] = float(np.var(fos_array, ddof=1))  # Sample Variance
            
            # 2. Distribution & Outlier Spread Analysis
            # Pearson's Median Skewness Coefficient = 3 * (Mean - Median) / Standard Deviation
            if self.analysis_results['std_dev_fos'] != 0:
                self.analysis_results['skewness_fos'] = 3 * (self.analysis_results['mean_fos'] - self.analysis_results['median_fos']) / self.analysis_results['std_dev_fos']
            else:
                self.analysis_results['skewness_fos'] = 0.0
                
            # Interquartile Range (IQR) implementation for Outlier Detection
            q75, q25 = np.percentile(fos_array, [75, 25])
            iqr = q75 - q25
            lower_bound = q25 - (1.5 * iqr)
            upper_bound = q75 + (1.5 * iqr)
            outliers = fos_array[(fos_array < lower_bound) | (fos_array > upper_bound)]
            self.analysis_results['outlier_count_fos'] = len(outliers)
            
            # 3. Structural Correlation Analysis (FoS response vs. Applied External Loading forces)
            correlation_matrix = np.corrcoef(fos_array, stress_array)
            self.analysis_results['correlation_fos_vs_stress'] = float(correlation_matrix[0, 1])
            
            # 4. Comparative Structural Group Analysis (Comparing Manufacturing/Testing Batches)
            if 'Batch_Group' in self.cleaned_data.columns:
                batches = self.cleaned_data['Batch_Group'].unique()
                if len(batches) >= 2:
                    batch1_fos = self.cleaned_data[self.cleaned_data['Batch_Group'] == batches[0]]['Factor_of_Safety'].to_numpy()
                    batch2_fos = self.cleaned_data[self.cleaned_data['Batch_Group'] == batches[1]]['Factor_of_Safety'].to_numpy()
                    self.analysis_results['comparative_analysis'] = {
                        f'Batch_{batches[0]}_Mean_FoS': float(np.mean(batch1_fos)),
                        f'Batch_{batches[1]}_Mean_FoS': float(np.mean(batch2_fos))
                    }
                    
            print("[SUCCESS] Engineering metrics successfully computed using NumPy arrays.")
            return self.analysis_results
            
        except Exception as e:
            print(f"[ERROR] Analytics computation failure: {e}")
            raise

    def generate_visualizations(self):
        """
        MODULE 4: VISUALIZATION SUITE (Hybrid Architecture)
        Generates 3 static plots via Seaborn/Matplotlib (100% immune to Kaleido browser crashes)
        and 2 animated interactive timelines using Plotly saved to clean HTML formats.
        """
        print("[INFO] Starting Visualization Module...")
        if self.cleaned_data is None:
            raise ValueError("No visualizable array structural data found.")
            
        df = self.cleaned_data
        sns.set_theme(style="whitegrid") # Apply clean engineering publication aesthetic
        
        # --- STATIC GRAPH 1: HISTOGRAM (Seaborn) ---
        plt.figure(figsize=(8, 5))
        sns.histplot(data=df, x='Factor_of_Safety', bins=25, kde=True, color='#1f77b4')
        plt.axvline(self.analysis_results['mean_fos'], color='r', linestyle='--', label=f"Mean: {self.analysis_results['mean_fos']:.2f}")
        plt.title('Distribution Profile of Safety Margins (SOL-05)')
        plt.xlabel('Calculated Factor of Safety (FoS)')
        plt.ylabel('Observation Sample Count')
        plt.legend()
        plt.tight_layout()
        plt.savefig("outputs/static_fos_histogram.png", dpi=300)
        plt.close()
        
        # --- STATIC GRAPH 2: BOX PLOT (Seaborn) ---
        plt.figure(figsize=(6, 5))
        sns.boxplot(data=df, y='Factor_of_Safety', color='#ff7f0e', width=0.4)
        plt.title('Statistical Variance & Outlier Detection Range')
        plt.ylabel('Factor of Safety Value')
        plt.tight_layout()
        plt.savefig("outputs/static_fos_boxplot.png", dpi=300)
        plt.close()
        
        # --- STATIC GRAPH 3: SCATTER PLOT (Matplotlib / Seaborn) ---
        plt.figure(figsize=(8, 5))
        scatter = plt.scatter(df['Applied_Stress'], df['Yield_Strength'], 
                              c=df['Factor_of_Safety'], cmap='plasma', alpha=0.8, edgecolors='none')
        plt.colorbar(scatter, label='Resulting Factor of Safety (FoS)')
        
        # Reference Line: Where Applied Stress equals Yield Strength (Boundary of Failure, FoS = 1.0)
        max_val = max(df['Applied_Stress'].max(), df['Yield_Strength'].max())
        plt.plot([0, max_val], [0, max_val], color='red', linestyle=':', label='Structural Failure Limit (FoS=1.0)')
        
        plt.title('Yield Structural Limits vs. Applied Operational Load')
        plt.xlabel('Applied Operational Stress (MPa)')
        plt.ylabel('Yield Strength Capacity Limit (MPa)')
        plt.legend()
        plt.tight_layout()
        plt.savefig("outputs/static_strength_vs_stress.png", dpi=300)
        plt.close()
        
        print("[SUCCESS] 3 Publication-grade Static Plots saved to /outputs folder.")

        # --- ANIMATED GRAPHS VIA PLOTLY (Saved directly as browser-ready HTML) ---
        time_col = 'Time_Step' if 'Time_Step' in df.columns else 'Simulated_Time_Step'
        df_sorted = df.sort_values(by=time_col)
        
        # Animation 1: Frequency distribution shifts over operational timelines
        fig_anim_hist = px.histogram(df_sorted, x='Factor_of_Safety', animation_frame=time_col,
                                     range_x=[float(df['Factor_of_Safety'].min()*0.8), float(df['Factor_of_Safety'].max()*1.2)],
                                     title='Dynamic Evolution of Structural Safety Profiles Over Operational Lifecycles',
                                     color_discrete_sequence=['#2ca02c'])
        fig_anim_hist.write_html("outputs/animated_fos_distribution.html")
        
        # Animation 2: Real-time degradation trajectory tracking
        fig_anim_scatter = px.scatter(df_sorted, x='Applied_Stress', y='Factor_of_Safety', animation_frame=time_col,
                                      range_x=[float(df['Applied_Stress'].min()*0.8), float(df['Applied_Stress'].max()*1.2)],
                                      range_y=[float(df['Factor_of_Safety'].min()*0.8), float(df['Factor_of_Safety'].max()*1.2)],
                                      title='Real-Time Operational Load Vector Trajectory vs. Structural Safety Runway Component')
        fig_anim_scatter.write_html("outputs/animated_stress_vs_fos.html")
        
        print("[SUCCESS] 2 Interactive Animated charts saved to /outputs folder.")


# --- AUTOMATED PIPELINE EXECUTION ENGINE ---
if __name__ == "__main__":
    target_data_file = "structural_data.csv"
    
    # Check if a dataset exists. If empty, generate a structural dataset to verify compilation.
    if not os.path.exists(target_data_file):
        print(f"[PRE-RUN] Local '{target_data_file}' not found. Generating compliant material matrix dataset...")
        np.random.seed(42)
        sample_size = 600
        mock_data = pd.DataFrame({
            'Material_Type': np.random.choice(['Steel-A36', 'Al-6061', 'Titanium-Grade5'], size=sample_size),
            'Yield_Strength': np.random.normal(loc=250.0, scale=12.0, size=sample_size), # Material Property
            'Applied_Stress': np.random.normal(loc=115.0, scale=22.0, size=sample_size), # Operational Load
            'Batch_Group': np.random.choice(['Batch_Alpha', 'Batch_Beta'], size=sample_size),
            'Time_Step': np.repeat(np.arange(1, 61), 10) # 60 Sequential structural testing cycles
        })
        mock_data.to_csv(target_data_file, index=False)
        print(f"[PRE-RUN] Synthesized local '{target_data_file}' dataset successfully.")

    # Instantiate the system architecture pipeline
    pipeline = EngineeringDataPipeline(file_path=target_data_file)
    
    # Run data system loop sequentially
    pipeline.ingest_data()
    pipeline.clean_data(target_material='Steel-A36') # Unique pipeline filter flag 
    computed_metrics = pipeline.analyze_data()
    pipeline.generate_visualizations()
    
    # Print execution results terminal summary table
    print("\n" + "="*50)
    print("       PIPELINE STATISTICS LOG COMPILATION SUMMARY")
    print("="*50)
    for key, val in computed_metrics.items():
        if isinstance(val, dict):
            print(f"{key.upper()}:")
            for sub_key, sub_val in val.items():
                print(f"  -> {sub_key}: {sub_val:.4f}")
        else:
            print(f"{key.upper()}: {val:.4f}" if isinstance(val, float) else f"{key.upper()}: {val}")
    print("="*50)

[INFO] Starting Data Ingestion Module...
[SUCCESS] Ingested 500 rows from raw source.
[INFO] Starting Data Cleaning & Unique Filtering Module...
[FILTER] Programmatic unique filter applied: Material_Type == 'Steel-A36'
[SUCCESS] Cleaning finished. Active evaluation slice contains 167 records.
[INFO] Starting Engineering Data Analytics Module...
[SUCCESS] Engineering metrics successfully computed using NumPy arrays.
[INFO] Starting Visualization Module...
[SUCCESS] 3 Publication-grade Static Plots saved to /outputs folder.
[SUCCESS] 2 Interactive Animated charts saved to /outputs folder.

       PIPELINE STATISTICS LOG COMPILATION SUMMARY
MEAN_FOS: 2.1720
MEDIAN_FOS: 2.0853
STD_DEV_FOS: 0.5638
VARIANCE_FOS: 0.3179
SKEWNESS_FOS: 0.4617
OUTLIER_COUNT_FOS: 7
CORRELATION_FOS_VS_STRESS: -0.9125
COMPARATIVE_ANALYSIS:
  -> Batch_Batch_B_Mean_FoS: 2.1577
  -> Batch_Batch_A_Mean_FoS: 2.1891
